In [ ]:
# Make sure you have the latest version of the SDK available to use the Batch API
!pip install openai --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.1/456.1 kB 8.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.59.6
    Uninstalling openai-1.59.6:
      Successfully uninstalled openai-1.59.6


In [ ]:
import json
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import ast
from IPython.display import Image, display
tqdm.pandas()

In [ ]:
client = OpenAI(api_key="Enter your Key")

# loading the data

In [ ]:
data=pd.read_excel('data/sample_data_fin_evals_exp.xlsx')
data.info()

#fact checking part I

In [ ]:
def get_fact_check_response(claim, evidence):
  prompt_fact_check = """
  You will receive a claim, and some trustworthy reliable information called evidence.  Your task is to divide the claim into multiple smaller subclaims/ atomic claims
  , then assess the factuality of each subclaim sentence based on the information provided in the evidence:

  * no error: the subclaim aligns explicitly with the content of the evidence and is factually consistent with it.
  * factuality error: the subclaim contains any factuality error.

  Instruction:
  First, compare each subclaim sentence with the evidence.
  Second, provide a single sentence explaining the factuality error in the subclaim and how to correct it based on the evidence.

  Provide your answer in JSON format. Your answer should strictly be a list of dictionaries whose keys are "sentence", "reason" and "correction". An example of your output should be:

  [{"sentence": "first  subclaim", "reason": "your reason", "correction": "your correction"},
   {"sentence": "second  subclaim", "reason": "your reason", "correction": "your correction"}]

  Claim:
  %s

  Evidence:
  %s
  """%(claim, evidence)
  response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    temperature=0,
    messages=[{"role": "user","content": prompt_fact_check}])
  print(response)
  result = response.choices[0].message.content.replace('json', '').replace('```', '').replace('\n', '').strip()
  try:
    result = json.loads(result)
  except:
    result=result+" check this result please!!!!!"
  return result


In [ ]:
get_fact_check_response('Moscow is in a country.', 'Moscow is the capital and most populous city of Russia , with 13.2 million residents within the city limits and 17.8 million within the urban area . Moscow has the status of a Russian federal city .')

ChatCompletion(id='chatcmpl-AsLvQ412YO4dJM4TLHuGqkcovWG1u', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n[\n  {"sentence": "Moscow is in a country.", "reason": "no error", "correction": ""}\n]\n```', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1737517356, model='gpt-4-1106-preview', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=31, prompt_tokens=290, total_tokens=321, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


[{'sentence': 'Moscow is in a country.',
  'reason': 'no error',
  'correction': ''}]

### creating batch file for the FinGrAct fact checking task

In [ ]:
tasks = []

for index, row in data.iterrows():
  prompt_fact_check = """
  You will receive a claim, and some trustworthy reliable information called evidence.  Your task is to divide the claim into multiple smaller subclaims/ atomic claims
  , then assess the factuality of each subclaim sentence based on the information provided in the evidence:

  * no error: the subclaim aligns explicitly with the content of the evidence and is factually consistent with it.
  * factuality error: the subclaim contains any factuality error.

  Instruction:
  First, compare each subclaim sentence with the evidence.
  Second, provide a single sentence explaining the factuality error in the subclaim and how to correct it based on the evidence.

  Provide your answer in JSON format. Your answer should strictly be a list of dictionaries whose keys are "sentence", "reason" and "correction". An example of your output should be:

  [{"sentence": "first erroneous subclaim", "reason": "your reason", "correction": "your correction"},
   {"sentence": "second erroneous subclaim", "reason": "your reason", "correction": "your correction"}]

  Claim:
  %s

  Evidence:
  %s
  """%(data.claim[index], data.evidence[index])


  task = {
        "custom_id": f"{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4-1106-preview",
            "temperature": 0,
            "messages": [
                {
                    "role": "user",
                    "content": prompt_fact_check
                }
            ],
        }
    }

  tasks.append(task)

In [ ]:
# Creating the file

file_name = "data/batch_FinGrAct_claims.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

### starting batch jobs

In [ ]:
# upload the file
batch_file = client.files.create(
  file=open(file_name, "rb"),
  purpose="batch"
)

In [ ]:
batch_job = client.batches.create(
  input_file_id=batch_file.id,
  endpoint="/v1/chat/completions",
  completion_window="24h"
)

In [ ]:
batch_job = client.batches.retrieve(batch_job.id)
print(batch_job) # check the status

Batch(id='batch_678fd008f74c819091777a28161b3f64', completion_window='24h', created_at=1737478160, endpoint='/v1/chat/completions', input_file_id='file-Mi3oxSsbw5ZbAVTHrFTmYH', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1737478511, error_file_id=None, errors=None, expired_at=None, expires_at=1737564560, failed_at=None, finalizing_at=1737478493, in_progress_at=1737478221, metadata=None, output_file_id='file-9JjdxUT5QFhUfQMqCxkJRr', request_counts=BatchRequestCounts(completed=203, failed=0, total=203))


###Retrieving results

In [ ]:
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content

In [ ]:
result_file_name = "data/batch_sample_FinGrAct_fact_check.jsonl"

with open(result_file_name, 'wb') as file:
    file.write(result)

In [ ]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [ ]:
res_df = pd.DataFrame(results)
res_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         203 non-null    object
 1   custom_id  203 non-null    object
 2   response   203 non-null    object
 3   error      0 non-null      object
dtypes: object(4)
memory usage: 6.5+ KB


In [ ]:
res_df.response[0]

{'status_code': 200,
 'request_id': '6cd3f05f031a8db725b2bda83b020b06',
 'body': {'id': 'chatcmpl-AsBkaJk6qEGekq8lpi2ORTQ8m1GHd',
  'object': 'chat.completion',
  'created': 1737478244,
  'model': 'gpt-4-1106-preview',
  'choices': [{'index': 0,
    'message': {'role': 'assistant',
     'content': '[{"sentence": "Anil Kapoor does not have a career.", "reason": "The subclaim is factually incorrect as the evidence states that Anil Kapoor is an actor and producer who has appeared in many Bollywood and international films, as well as television series, which constitutes a career.", "correction": "Anil Kapoor has a career as an actor and producer in Bollywood and international films, as well as television series."}]',
     'refusal': None},
    'logprobs': None,
    'finish_reason': 'stop'}],
  'usage': {'prompt_tokens': 310,
   'completion_tokens': 89,
   'total_tokens': 399,
   'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
   'completion_tokens_details': {'reasoning_to

In [ ]:
def get_response(input_dict):
  return json.loads(input_dict['body']['choices'][0]['message']['content'].replace('json', '').replace('```', '').replace('\n', '').replace('  ',' ').strip())

In [ ]:
res_df['FinGrAct_res'] = res_df['response'].progress_apply(get_response)

100%|██████████| 203/203 [00:00<00:00, 22088.46it/s]


In [ ]:
resu = res_df[['custom_id', 'FinGrAct_res']]
resu.to_excel('data/FinGrAct_response.xlsx', index=False)

### checking the results

In [ ]:
res=pd.read_excel('data/sample_data_fin_evals.xlsx')
res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   claim                   203 non-null    object
 1   evidence                203 non-null    object
 2   sources                 87 non-null     object
 3   label                   203 non-null    bool  
 4   metadata                203 non-null    object
 5   explanation             203 non-null    object
 6   Human_annotation_exp    203 non-null    object
 7   mistral_exp             203 non-null    object
 8   Human_annotation_exp.1  203 non-null    object
 9   llama_exp               203 non-null    object
 10  Human_annotation_exp.2  203 non-null    object
 11  gpt-4_exp               203 non-null    object
 12  Human_annotation_exp.3  203 non-null    object
 13  geval_eval              203 non-null    object
 14  prom_eval               203 non-null    object
 15  prom_e

In [ ]:
data=pd.read_excel('data/FinGrAct_response.xlsx')
data.info()

In [ ]:
indices=[]
for i in range(len(data)):
  if len(str(data.FinGrAct_res[i]).replace('[','').replace(']',''))==0:
    indices.append(data.custom_id[i])
indices

[]

In [ ]:
len(indices)

17

In [ ]:
for i in tqdm(indices):
  data.FinGrAct_res[i] = get_fact_check_response(res.claim[i], res.evidence[i])
  time.sleep(2)

In [ ]:
data.to_excel('data/FinGrAct_response.xlsx', index=False)

In [ ]:
res.evidence[34]

"It premiered on August 11 , 1991 as one of the original three Nicktoons , along with Rugrats and Doug . The show ended on December 16 , 1995 , with a total of five seasons and 52 episodes . The Ren & Stimpy Show , often simply called Ren & Stimpy , is an American children 's / adult animated television series created by John Kricfalusi for Nickelodeon ."

# alignment task for the FinGrAct adaptation

In [ ]:
data=pd.read_excel('data/sample_data_fin_evals.xlsx')
data.info()

In [ ]:
data.FinGrAct_fact_check_res[0]

"[{'sentence': 'Anil Kapoor does not have a career.', 'reason': 'The subclaim is factually incorrect as the evidence states that Anil Kapoor is an actor and producer who has appeared in many Bollywood and international films, as well as television series, which constitutes a career.', 'correction': 'Anil Kapoor has a career as an actor and producer in Bollywood and international films, as well as television series.'}]"

In [ ]:
def extract_subclaims(input_dict):
  ans_list=ast.literal_eval(input_dict)
  error_list=[ans_list[i]['reason'] for i in range(len(ans_list))]
  correction_list=[ans_list[i]['correction'] for i in range(len(ans_list))]
  return error_list, correction_list

In [ ]:
extract_subclaims(data.FinGrAct_fact_check_res[0])

(['The subclaim is factually incorrect as the evidence states that Anil Kapoor is an actor and producer who has appeared in many Bollywood and international films, as well as television series, which constitutes a career.'],
 ['Anil Kapoor has a career as an actor and producer in Bollywood and international films, as well as television series.'])

In [ ]:
def fact_alignment(input_dict, explanation:str, src:str):
  subclaim_list, corrections_list = extract_subclaims(input_dict)
  prompt_act_evaluate ='''
  You will receive a list of errors, their corrections, and a transcript called the 'explanation'. Your task is to assess if each of the errors can be inferred from the explanation,
  and if the corrections can be inferred from the explanation as well.

  Instruction:
  First, compare each error with the explanation.
  Second, check if the error is inferred from the explanation and then response "Yes" or "No" for each detected error explicitly mentioned in the explanation.
  Third, compare each correction with the explanation.
  Fourth, check if the correction is inferred from the explanation and then respond with "Yes" or "No" for each detected correction.
  Fifth, based on your knowledge, check if there are credible and relevent web links in the explanation supporting it, and then respond with "Yes" or "No" for each detected relevent web link.

  Provide your answer in JSON format. The answer should be a list of dictionaries whose keys are "error", "response", "correction", and "supporting_links". An example of your output:
  [{"error": "first error", "response": "Yes", "correction": "Yes", "supporting_links": "Yes"}, {"error": "second error", "response": "No", "correction": "Yes", "supporting_links": "No"}, {"error": "third error", "response": "Yes", "correction": "No", "supporting_links": "No"}]

  list of errors:
  %s

  corrections:
  %s

  explanation:
  %s
  %s

  '''%(subclaim_list, corrections_list, explanation, src)
  response = client.chat.completions.create(
  model="gpt-4-1106-preview",
  temperature=0,
  messages=[{"role": "user","content": prompt_act_evaluate}])
  print(response)
  result = response.choices[0].message.content.replace('json', '').replace('```', '').replace('\n', '').strip()
  try:
    result = json.loads(result)
  except:
    result=result+" check this result please!!!!!"
  return result

In [ ]:
tasks = []

for index, row in data.iterrows():
  subclaim_list, corrections_list = extract_subclaims(data.FinGrAct_fact_check_res[index])
  prompt_act_evaluate ='''
  You will receive a list of errors, their corrections, and a transcript called the 'explanation'. Your task is to assess if each of the errors can be inferred from the explanation,
  and if the corrections can be inferred from the explanation as well.

  Instruction:
  First, compare each error with the explanation.
  Second, check if the error is inferred from the explanation and then response "Yes" or "No" for each detected error explicitly mentioned in the explanation.
  Third, compare each correction with the explanation.
  Fourth, check if the correction is inferred from the explanation and then respond with "Yes" or "No" for each detected correction.
  Fifth, check if there are credible and relevent web links in the explanation supporting it, and then respond with "Yes" or "No" for each detected relevent web link.

  Provide your answer in JSON format. The answer should be a list of dictionaries whose keys are "error", "response", "correction", and "supporting_links". An example of your output:
  [{"error": "first error", "response": "Yes", "correction": "Yes", "supporting_links": "Yes"}, {"error": "second error", "response": "No", "correction": "Yes", "supporting_links": "No"}, {"error": "third error", "response": "Yes", "correction": "No", "supporting_links": "No"}]

  list of errors:
  %s

  corrections:
  %s

  explanation:
  %s
  '''%(subclaim_list, corrections_list, data['gpt-4_exp'][index])


  task = {
        "custom_id": f"{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4-1106-preview",
            "temperature": 0,
            "messages": [
                {
                    "role": "user",
                    "content": prompt_act_evaluate
                }
            ],
        }
    }

  tasks.append(task)

In [ ]:
tasks[0]

{'custom_id': '0',
 'method': 'POST',
 'url': '/v1/chat/completions',
 'body': {'model': 'gpt-4-1106-preview',
  'temperature': 0,
  'messages': [{'role': 'user',
    'content': '\n  You will receive a list of errors, their corrections, and a transcript called the \'explanation\'. Your task is to assess if each of the errors can be inferred from the explanation,\n  and if the corrections can be inferred from the explanation as well.\n\n  Instruction:\n  First, compare each error with the explanation.\n  Second, check if the error is inferred from the explanation and then response "Yes" or "No" for each detected error explicitly mentioned in the explanation.\n  Third, compare each correction with the explanation.\n  Fourth, check if the correction is inferred from the explanation and then respond with "Yes" or "No" for each detected correction.\n  Fifth, check if there are credible and relevent web links in the explanation supporting it, and then respond with "Yes" or "No" for each dete

In [ ]:
# Creating the file

file_name = "data/batch_FinGrAct_alignments_gpt4.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

### starting batch jobs

In [ ]:
# upload the file
batch_file = client.files.create(
  file=open(file_name, "rb"),
  purpose="batch"
)

In [ ]:
batch_job = client.batches.create(
  input_file_id=batch_file.id,
  endpoint="/v1/chat/completions",
  completion_window="24h"
)

In [ ]:
batch_job = client.batches.retrieve(batch_job.id)
print(batch_job) # check the status

Batch(id='batch_67907ac2d3a8819088b51d404eb0626f', completion_window='24h', created_at=1737521858, endpoint='/v1/chat/completions', input_file_id='file-DCQVB5LHVrHhgJ6qnUrHDN', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1737522047, error_file_id=None, errors=None, expired_at=None, expires_at=1737608258, failed_at=None, finalizing_at=1737522031, in_progress_at=1737521860, metadata=None, output_file_id='file-7XRuHdidExnTd4e6LuTYXY', request_counts=BatchRequestCounts(completed=203, failed=0, total=203))


###Retrieving results

In [ ]:
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content

In [ ]:
result_file_name = "data/batch_sample_FinGrAct_align_gpt_exp.jsonl"

with open(result_file_name, 'wb') as file:
    file.write(result)

In [ ]:
# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [ ]:
res_df = pd.DataFrame(results)
res_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         203 non-null    object
 1   custom_id  203 non-null    object
 2   response   203 non-null    object
 3   error      0 non-null      object
dtypes: object(4)
memory usage: 6.5+ KB


In [ ]:
res_df.head()

,id,custom_id,response,error
0,batch_req_679074c9c1e48190b2511b4f544d31c3,0,"{'status_code': 200, 'request_id': '3f44ac4b91...",None
1,batch_req_679074c9e45c8190a615dc1ccb7a2026,1,"{'status_code': 200, 'request_id': 'ebc3628bba...",None
2,batch_req_679074ca07508190b055854ffc4f0273,2,"{'status_code': 200, 'request_id': 'bde60779b9...",None
3,batch_req_679074ca29b88190a6333808eddd99a0,3,"{'status_code': 200, 'request_id': 'ecd146f11d...",None
4,batch_req_679074ca4cb08190a9a79031824f9b07,4,"{'status_code': 200, 'request_id': 'c68739b6ad...",None


## processing FinGrAct alignment results

In [ ]:
# Loading data from saved file
results = []
with open("./data/batch_sample_FinGrAct_align_mistral_exp.jsonl", 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [ ]:
results[0]

{'id': 'batch_req_6790788de8e08190a21afc94f805b287',
 'custom_id': '0',
 'response': {'status_code': 200,
  'request_id': '5cd8dd95229251e40c110cd31867bbf2',
  'body': {'id': 'chatcmpl-AsMuEagQcos7aLpHd6yZxX7nhC0Qb',
   'object': 'chat.completion',
   'created': 1737521126,
   'model': 'gpt-4-1106-preview',
   'choices': [{'index': 0,
     'message': {'role': 'assistant',
      'content': '```json\n[\n  {\n    "error": "The subclaim is factually incorrect as the evidence states that Anil Kapoor is an actor and producer who has appeared in many Bollywood and international films, as well as television series, which constitutes a career.",\n    "response": "Yes",\n    "correction": "Yes",\n    "supporting_links": "Yes"\n  }\n]\n```',
      'refusal': None},
     'logprobs': None,
     'finish_reason': 'stop'}],
   'usage': {'prompt_tokens': 533,
    'completion_tokens': 81,
    'total_tokens': 614,
    'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
    'completion_token

In [ ]:
res_df = pd.DataFrame(results)[["custom_id","response"]]
res_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   custom_id  203 non-null    object
 1   response   203 non-null    object
dtypes: object(2)
memory usage: 3.3+ KB


In [ ]:
def get_response(input_dict):
  return json.loads(input_dict['body']['choices'][0]['message']['content'].replace('json', '').replace('```', '').replace('\n', '').replace('  ',' ').strip())

In [ ]:
res_df["FinGrAct_align_mistral_exp"] = res_df['response'].progress_apply(get_response)

100%|██████████| 203/203 [00:00<00:00, 28542.25it/s]


In [ ]:
data["FinGrAct_align_mistral_exp"]=res_df.FinGrAct_align_mistral_exp.to_list()

In [ ]:
data.info()

In [ ]:
data.to_excel("data/sample_data_fin_evals_exp.xlsx", index=False)

### getting the scores for FinGrAct

In [ ]:
def get_FinGrAct_score(input_dict):
  ans_list=ast.literal_eval(input_dict)
  supporting_links= False
  num_fact_errors_detected, num_corrections = 0, 0
  num_subclaims = len(ans_list)
  for item in ans_list:
    if item['supporting_links']=='Yes':
      supporting_links=True
    if item['response']=='Yes':
      num_fact_errors_detected+=1
    if item['correction']=='Yes':
      num_corrections+=1
  error_detction_perc, error_correction_perc, supporting_links_perc = num_fact_errors_detected/num_subclaims, num_corrections/num_subclaims, supporting_links/num_subclaims
  if error_detction_perc == 0:
    error_detection = 0
  elif error_detction_perc < 1:
    error_detection = 1
  else:
    error_detection = 2
  if error_correction_perc == 0:
    error_correction = 0
  elif error_correction_perc < 1:
    error_correction = 1
  else:
    error_correction = 2
  if supporting_links_perc ==0:
    supporting_links = 0
  elif supporting_links_perc < 1:
    supporting_links = 1
  else:
    supporting_links = 2
  return error_detection, error_correction, supporting_links

In [ ]:
data.info()

In [ ]:
FinGrAct_scores=[]
FinGrAct_likert_scale=[]
for index in range(len(data)):
  FinGrAct_exp_score = get_FinGrAct_score(data.FinGrAct_align_exp[index])
  FinGrAct_mistral_score = get_FinGrAct_score(data.FinGrAct_align_mistral_exp[index])
  FinGrAct_llama_score = get_FinGrAct_score(data.FinGrAct_align_llama_exp[index])
  FinGrAct_gpt_score = get_FinGrAct_score(data.FinGrAct_align_gpt_exp[index])
  FinGrAct_scores.append([FinGrAct_exp_score, FinGrAct_mistral_score, FinGrAct_llama_score, FinGrAct_gpt_score])

In [ ]:
data['FinGrAct_detailed_eval'] = FinGrAct_scores

In [ ]:
for row in FinGrAct_scores:
  FinGrAct_likert=[]
  for item in row:
    FinGrAct_likert.append(round(sum(list(item))*(5/6)))
  FinGrAct_likert_scale.append(FinGrAct_likert)

In [ ]:
FinGrAct_likert_scale[:5]

[[5, 5, 5, 5], [2, 3, 5, 3], [3, 3, 3, 3], [3, 3, 3, 2], [0, 3, 3, 3]]

In [ ]:
data.FinGrAct_eval = FinGrAct_likert_scale

In [ ]:
data.to_excel("data/sample_data_fin_evals_exp.xlsx", index=False)